# Chat interface

`gr.ChatInterface` calls `chat(message, history)` and keeps the transcript in the page. The model still needs that transcript in `messages` on every turn.

## Clients

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

azure = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
azure_model = os.getenv("GPT_MODEL", "gpt-4o")

import gradio as gr


## History

In [2]:
def messages_for(message, history, instructions):
    history = [{"role": turn["role"], "content": turn["content"]} for turn in history]
    return (
        [{"role": "system", "content": instructions}]
        + history
        + [{"role": "user", "content": message}]
    )


sample = messages_for(
    "Do you have hats?",
    [{"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Welcome in."}],
    "You are a helpful assistant.",
)
sample


[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Welcome in.'},
 {'role': 'user', 'content': 'Do you have hats?'}]

## Streaming callback

In [3]:
def stream_chat(message, history, instructions):
    stream = azure.chat.completions.create(
        model=azure_model,
        messages=messages_for(message, history, instructions),
        stream=True,
        timeout=120.0,
    )
    response = ""
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            yield response


## One-shot prompt

In [4]:
store_system = """
You are a helpful assistant in a clothes store. Gently point customers toward items on sale.
Hats are 60% off, and most other items are 50% off.
For example, if the customer says "I'm looking to buy a hat", you could reply
"We have a lot of hats, including several in the sale."
If they are unsure what to get, suggest a hat.
Respond in markdown without code blocks.
"""


def store_chat(message, history):
    instructions = store_system
    if "belt" in message.lower():
        instructions += (
            " The store does not sell belts. If you are asked for belts, "
            "point out other items on sale."
        )
    yield from stream_chat(message, history, instructions)


gr.ChatInterface(fn=store_chat, title="Clothes store").launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Business assistant

In [5]:
studio_system = """
You are the client assistant for a small machine-learning consultancy.
The consultancy builds NLP applications, prediction models, and the backend services around them.
Speak in a calm, practical tone. Use short paragraphs and markdown.
Offer a sensible next step, such as a scoping call, rather than a hard sell.

Example:
User: Can you build a chatbot for our support team?
Assistant: Yes. A useful first version is a support assistant with your help-center
articles in context, plus a clear handoff to a person. A 45-minute scoping call
is enough to see whether that fits.
"""


def studio_chat(message, history):
    instructions = studio_system
    lowered = message.lower()
    if "price" in lowered or "cost" in lowered or "quote" in lowered:
        instructions += (
            " Scoping calls are free. A small prototype is usually a fixed two-week "
            "engagement. Do not invent a precise quote."
        )
    if "hire" in lowered or "job" in lowered or "career" in lowered:
        instructions += (
            " We are not hiring through this chat. Ask them to email a short introduction."
        )
    yield from stream_chat(message, history, instructions)


gr.ChatInterface(fn=studio_chat, title="Consultancy assistant").launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
